# FlashInfer CUDA 13 compiler recovery

Fresh Kaggle: enable Internet + T4, then Run All. Return the final ZIP even if it fails.

The returned run passed 808 CPU checks. Its first native build failed because
CUDA 13 made template launch stubs local to individual source files; the older
FlashInfer kernel expects to link them across files. Subsequent missing-library
errors follow from that failed build. No native attention result was collected.

This replay keeps the original source snapshot byte-for-byte, including all
decoder math, FlashInfer 0.2.5, tests, thresholds and retained failure control.
It detects the actual NVCC release (not Torch's CUDA runtime version) and adds
`--static-global-template-stub=false` on CUDA 13+. Each run uses a fresh isolated
FlashInfer build directory. No package source is patched and no old cache is deleted.

A one-key build/load smoke runs first, then the same eight native tests, 15+1
comparison, frozen paging suite and cached trajectory. The compiler fix remains
unvalidated on T4 until execution. P5.3 and whole-model acceptance stay open.


In [ ]:
import base64
import hashlib
import io
import json
import os
import subprocess
import sys
import tempfile
import traceback
import zipfile
from pathlib import Path

from IPython.display import FileLink, display

FLASHINFER_URL = "https://files.pythonhosted.org/packages/b2/c4/9ec0f79e2480fc5c93307c4a1ac903e5cf33c551c0eaeb648196234b55af/flashinfer_python-0.2.5.tar.gz"
FLASHINFER_SHA256 = "990aa090ef781783e76b836696ece4efd23956f72b5696d622fc619a61162aef"
TOOLCHAIN_HELPER_SOURCE = 'def recovery_environment(nvcc_output, workspace_base, existing_append_flags=""):\n    """Use the actual NVCC release, independently of Torch\'s CUDA runtime version.\n\n    This function is embedded unchanged in the replay notebook, so imports stay\n    local. NVCC_APPEND_FLAGS is NVIDIA\'s compiler hook, not a FlashInfer patch.\n    """\n    import re\n    import shlex\n    from pathlib import Path\n\n    match = re.search(r"Cuda compilation tools,\\s+release (\\d+)\\.(\\d+)", nvcc_output)\n    if match is None:\n        raise ValueError("cannot identify the actual NVCC release")\n    release = tuple(map(int, match.groups()))\n    flags = existing_append_flags.strip()\n    selected = []\n    if release >= (13, 0):\n        tokens = shlex.split(flags)\n        settings = []\n        for index, token in enumerate(tokens):\n            name, _, value = token.partition("=")\n            if name in ("-static-global-template-stub", "--static-global-template-stub"):\n                settings.append(\n                    value if "=" in token else tokens[index + 1] if index + 1 < len(tokens) else ""\n                )\n        if any(setting != "false" for setting in settings):\n            raise ValueError("existing NVCC flags conflict with the legacy template-stub recovery")\n        if not settings:\n            selected = ["--static-global-template-stub=false"]\n            flags = (flags + " " + selected[0]).strip()\n    overrides = {\n        "FLASHINFER_WORKSPACE_BASE": str(Path(workspace_base).resolve()),\n        "FLASHINFER_JIT_VERBOSE": "1",\n    }\n    if flags:\n        overrides["NVCC_APPEND_FLAGS"] = flags\n    return {\n        "nvcc_release": list(release),\n        "added_nvcc_flags": selected,\n        "environment_overrides": overrides,\n        "scope": "Compiler linkage only; same pinned FlashInfer and decoder arithmetic",\n    }\n'
FLASHINFER_WARMUP = 'import json, sys, traceback\nfrom pathlib import Path\nimport torch\nfrom pagedcore.engine_types import CapacityCounts, KVReservation, RequestId\nfrom pagedcore.flashinfer_backend import FlashInferAttention\nfrom pagedcore.kv_cache import KVAllocator\n\nreport = {"scope": "One-key native build/load smoke; not model accuracy acceptance"}\nallocator = None\nrequest = RequestId("compile-smoke")\ntry:\n    backend = FlashInferAttention(device="cuda:0")\n    pool = torch.full((22, 2, 1, 16, 4, 64), torch.nan, dtype=torch.float16, device="cuda:0")\n    allocator = KVAllocator(pool)\n    allocator.create(KVReservation(request, 1))\n    key = torch.zeros((22, 1, 4, 64), dtype=torch.float16, device=pool.device)\n    value = torch.ones_like(key)\n    allocator.append(request, key, value)\n    query = torch.zeros((1, 32, 1, 64), dtype=torch.float16, device=pool.device)\n    output = backend(query, allocator, request, layer_index=0)\n    torch.cuda.synchronize()\n    torch.testing.assert_close(output, torch.ones_like(output), atol=0, rtol=0)\n    report.update(native_library_loaded=True, one_key_output_exact=True)\nexcept BaseException:\n    report["error"] = traceback.format_exc()\n    raise\nfinally:\n    if allocator is not None:\n        allocator.release_all(request)\n        assert allocator.capacity() == CapacityCounts(1, 1, 0, 0, 0, 0)\n        report["cleanup_complete"] = True\n    Path(sys.argv[1]).write_text(json.dumps(report, indent=2))\n'
TOOLCHAIN_REPLAY_ORIGIN = {
    "returned_archive": "pagedcore-flashinfer-7behvy1m.zip",
    "returned_archive_sha256": "9211c219924908fffeb6ec010f8b649f9f05f80498de65be9e29fc23a3336c2f",
    "original_notebook_sha256": "ebf1b4829a3ef0b2e9b3dc644c965a89f8c736c3a366d2c244782967a8a7e784",
    "source_snapshot_sha256": "f3eaa51ecfd13bc1ed9f333f3a320f91d0e27bc3f93f41d54ee24544fba2b576",
    "source_snapshot_unchanged": True,
    "helper_sha256": "ccfdc4a140b7ee0e6af547efb000ea2df616763254d2aed2949e3855d723b51b",
    "warmup_sha256": "ae78eecb23654def92b17067f57bad134f8e63d2175b532285ac2cd1f9492ea0",
}


def recovery_environment(nvcc_output, workspace_base, existing_append_flags=""):
    """Use the actual NVCC release, independently of Torch's CUDA runtime version.

    This function is embedded unchanged in the replay notebook, so imports stay
    local. NVCC_APPEND_FLAGS is NVIDIA's compiler hook, not a FlashInfer patch.
    """
    import re
    import shlex
    from pathlib import Path

    match = re.search(r"Cuda compilation tools,\s+release (\d+)\.(\d+)", nvcc_output)
    if match is None:
        raise ValueError("cannot identify the actual NVCC release")
    release = tuple(map(int, match.groups()))
    flags = existing_append_flags.strip()
    selected = []
    if release >= (13, 0):
        tokens = shlex.split(flags)
        settings = []
        for index, token in enumerate(tokens):
            name, _, value = token.partition("=")
            if name in ("-static-global-template-stub", "--static-global-template-stub"):
                settings.append(
                    value if "=" in token else tokens[index + 1] if index + 1 < len(tokens) else ""
                )
        if any(setting != "false" for setting in settings):
            raise ValueError("existing NVCC flags conflict with the legacy template-stub recovery")
        if not settings:
            selected = ["--static-global-template-stub=false"]
            flags = (flags + " " + selected[0]).strip()
    overrides = {
        "FLASHINFER_WORKSPACE_BASE": str(Path(workspace_base).resolve()),
        "FLASHINFER_JIT_VERBOSE": "1",
    }
    if flags:
        overrides["NVCC_APPEND_FLAGS"] = flags
    return {
        "nvcc_release": list(release),
        "added_nvcc_flags": selected,
        "environment_overrides": overrides,
        "scope": "Compiler linkage only; same pinned FlashInfer and decoder arithmetic",
    }


PAYLOAD = "..."
SNAPSHOT_SHA256 = "f3eaa51ecfd13bc1ed9f333f3a320f91d0e27bc3f93f41d54ee24544fba2b576"


output_override = os.environ.get("PAGEDCORE_EVIDENCE_WORKDIR")
working = (
    Path(output_override).expanduser().resolve()
    if output_override
    else Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path.cwd()
)
working.mkdir(parents=True, exist_ok=True)
run_root = Path(tempfile.mkdtemp(prefix="pagedcore-flashinfer-", dir=working))
source = run_root / "source"
evidence = run_root / "evidence"
evidence.mkdir()
payload = base64.b64decode(PAYLOAD)
assert hashlib.sha256(payload).hexdigest() == SNAPSHOT_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(source)
identity = json.loads((source / "source-identity.json").read_text())
environment = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="0",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    TOKENIZERS_PARALLELISM="false",
)
commands = []


def run(name, command, *, required=False):
    record = {"name": name, "argv": list(map(str, command)), "cwd": str(source)}
    commands.append(record)
    print(name, flush=True)
    with (evidence / (name + ".log")).open("w") as log:
        process = subprocess.Popen(
            command, cwd=source, env=environment, stdout=log, stderr=subprocess.STDOUT
        )
        try:
            while process.poll() is None:
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    print(name + " is running; progress is saved in " + str(log.name), flush=True)
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            record["interrupted"] = True
            raise
    record["returncode"] = process.returncode
    (evidence / "commands.json").write_text(json.dumps(commands, indent=2))
    print(name, "exit", process.returncode, flush=True)
    if required and process.returncode:
        raise RuntimeError(name + " failed; inspect its retained log")


def verify_source():
    checks = {
        name: hashlib.sha256((source / name).read_bytes()).hexdigest()
        for name in identity["sha256"]
    }
    assert checks == identity["sha256"], "source changed after notebook generation"
    return checks


setup_ok = False
try:
    verify_source()
    run("setup-uv", [sys.executable, "-m", "pip", "install", "uv==0.11.27"], required=True)
    run(
        "setup-frozen",
        [
            sys.executable,
            "-m",
            "uv",
            "sync",
            "--frozen",
            "--extra",
            "cuda",
            "--extra",
            "dev",
            "--python",
            (source / ".python-version").read_text().strip(),
        ],
        required=True,
    )
    python = source / ".venv/bin/python"
    run(
        "resolved-dependencies",
        [sys.executable, "-m", "uv", "pip", "freeze", "--python", str(python)],
    )
    run("gpu-inventory", ["nvidia-smi", "-q"])

    target_code = "import json,sys,torch; from pathlib import Path; from pagedcore.cli import collect_environment_diagnostics; assert torch.cuda.is_available() and torch.cuda.device_count()==1; assert 'T4' in torch.cuda.get_device_name(0); assert torch.__version__=='2.7.1+cu126'; p=torch.cuda.get_device_properties(0); Path(sys.argv[1]).write_text(json.dumps(dict(torch=torch.__version__,name=p.name,uuid=str(p.uuid),capability=[p.major,p.minor],visible_devices=torch.cuda.device_count(),total_memory=p.total_memory,environment=collect_environment_diagnostics()),indent=2))"
    run("target", [str(python), "-c", target_code, str(evidence / "target.json")], required=True)
    run(
        "checkpoint-verify",
        [
            str(python),
            "-m",
            "scripts.verify_owned_checkpoint",
            "--output",
            str(evidence / "environment-manifest.json"),
        ],
        required=True,
    )

    environment["TORCH_CUDA_ARCH_LIST"] = "7.5"
    environment["MAX_JOBS"] = "2"
    run("nvcc", ["nvcc", "--version"], required=True)
    recovery = recovery_environment(
        (evidence / "nvcc.log").read_text(),
        run_root / "nvcc13-fresh-workspace",
        environment.get("NVCC_APPEND_FLAGS", ""),
    )
    environment.update(recovery["environment_overrides"])
    (evidence / "toolchain-recovery.py").write_text(TOOLCHAIN_HELPER_SOURCE)
    (evidence / "native-warmup.py").write_text(FLASHINFER_WARMUP)
    (evidence / "toolchain-recovery.json").write_text(
        json.dumps({**TOOLCHAIN_REPLAY_ORIGIN, **recovery}, indent=2)
    )
    run("compiler", ["c++", "--version"], required=True)
    run(
        "setup-ninja",
        [
            sys.executable,
            "-m",
            "uv",
            "pip",
            "install",
            "--python",
            str(python),
            "--no-deps",
            "ninja==1.11.1.4",
        ],
        required=True,
    )
    download = """
import hashlib, sys, urllib.request
from pathlib import Path
data = urllib.request.urlopen(sys.argv[1], timeout=120).read()
assert hashlib.sha256(data).hexdigest() == sys.argv[2], 'FlashInfer source checksum mismatch'
Path(sys.argv[3]).write_bytes(data)
"""
    artifact = evidence / "flashinfer_python-0.2.5.tar.gz"
    run(
        "download-flashinfer",
        [str(python), "-c", download, FLASHINFER_URL, FLASHINFER_SHA256, str(artifact)],
        required=True,
    )
    run(
        "setup-flashinfer",
        [
            sys.executable,
            "-m",
            "uv",
            "pip",
            "install",
            "--python",
            str(python),
            "--no-deps",
            "--no-build-isolation",
            str(artifact),
        ],
        required=True,
    )
    run(
        "candidate-dependencies",
        [sys.executable, "-m", "uv", "pip", "freeze", "--python", str(python)],
        required=True,
    )
    run(
        "candidate-dependency-check",
        [sys.executable, "-m", "uv", "pip", "check", "--python", str(python)],
        required=True,
    )
    installed = """
import json, sys, torch
from importlib.metadata import version
from pathlib import Path
assert torch.__version__ == '2.7.1+cu126', 'candidate install changed Torch'
assert version('flashinfer-python') == '0.2.5'
Path(sys.argv[1]).write_text(json.dumps(dict(torch=torch.__version__,
    flashinfer=version('flashinfer-python'), ninja=version('ninja'),
    cuda=torch.version.cuda, source_sha256=sys.argv[2]), indent=2))
"""
    run(
        "candidate-identity",
        [str(python), "-c", installed, str(evidence / "candidate.json"), FLASHINFER_SHA256],
        required=True,
    )

    setup_ok = True
except BaseException:
    (evidence / "setup-failure.txt").write_text(traceback.format_exc())
    print("Setup incomplete; run the final cell to download its logs.")

In [ ]:
checks_ok = False
try:
    assert setup_ok, "Setup did not finish"
    run("lint", [str(python), "-m", "ruff", "check", "."], required=True)
    run("format", [str(python), "-m", "ruff", "format", "--check", "."], required=True)
    run("types", [str(python), "-m", "mypy", "--strict", "src/pagedcore"], required=True)
    run(
        "cpu-tests",
        [
            str(python),
            "-m",
            "pytest",
            "-m",
            "not gpu and not benchmark",
            "-ra",
            "--junitxml=" + str(evidence / "cpu-tests.xml"),
        ],
        required=True,
    )
    checks_ok = True
except BaseException:
    (evidence / "checks-failure.txt").write_text(traceback.format_exc())
    print("Checks incomplete; keep running to the export cell.")

In [ ]:
kernel_ok = False
try:
    assert checks_ok, "CPU checks did not finish"
    run(
        "native-compile-smoke",
        [str(python), str(evidence / "native-warmup.py"), str(evidence / "native-warmup.json")],
        required=True,
    )
    run(
        "native-kernel-tests",
        [
            str(python),
            "-m",
            "pytest",
            "tests/gpu/test_flashinfer_cuda.py",
            "--decode-backend=flashinfer",
            "-v",
            "-ra",
            "--junitxml=" + str(evidence / "native-kernel-tests.xml"),
        ],
        required=True,
    )
    kernel_ok = True
except BaseException:
    (evidence / "kernel-failure.txt").write_text(traceback.format_exc())
    print("Native kernel checks incomplete; export the failure logs.")

In [ ]:
try:
    assert kernel_ok, "Native kernel checks did not pass"
    for backend in ("pagedcore", "flashinfer"):
        run(
            "focused-15-plus-1-" + backend,
            [
                str(python),
                "-m",
                "scripts.probe_two_token_decode",
                "--prompt-tokens",
                "15",
                "--single-only",
                "--attention-backend",
                backend,
                "--output-dir",
                str(evidence / ("focused-" + backend)),
            ],
            required=True,
        )
    control = """
import json, sys, torch
from pathlib import Path
root, output = map(Path, sys.argv[1:])
prior = torch.load(root / 'prior-failure/target.pt', map_location='cpu', weights_only=True)
current = torch.load(output / 'focused-pagedcore/single/target.pt', map_location='cpu', weights_only=True)
controls = {name + '_exact': torch.equal(prior[name], current[name]) for name in prior}
(output / 'prior-failure-controls.json').write_text(json.dumps(controls, indent=2))
assert all(controls.values()), 'Baseline changed; do not interpret the candidate as a controlled improvement'
"""
    run(
        "prior-failure-control",
        [str(python), "-c", control, str(source), str(evidence)],
        required=True,
    )
    for backend in ("pagedcore", "flashinfer"):
        result = json.loads((evidence / ("focused-" + backend) / "single/report.json").read_text())
        print(
            backend,
            "logit ratio:",
            result["owned_reference"]["max_tolerance_ratio"],
            "greedy agrees:",
            result["owned_reference"]["greedy_matches"],
        )
    # Numerical failures remain recorded; collect the whole frozen suite regardless.
    environment["PAGEDCORE_ACCURACY_DIR"] = str(evidence / "raw-logits")
    run(
        "gpu-regressions",
        [
            str(python),
            "-m",
            "pytest",
            "-m",
            "gpu",
            "--decode-backend=flashinfer",
            "-v",
            "-ra",
            "--junitxml=" + str(evidence / "gpu-tests.xml"),
        ],
    )
    run("accuracy-export", [str(python), "-m", "scripts.export_paged_accuracy", str(evidence)])
    run(
        "candidate-cached-trajectory",
        [
            str(python),
            "-m",
            "scripts.check_checkpoint_trajectory",
            "--attention-backend",
            "flashinfer",
            "--output-dir",
            str(evidence / "trajectory"),
        ],
        required=True,
    )
    summary = json.loads((evidence / "accuracy-summary.json").read_text())
    if summary["status"] == "passed":
        assert all(c["attention_backend"] == "flashinfer-0.2.5" for c in summary["cases"].values())
    print("Frozen paging accuracy:", summary["status"])
    print(
        "Return the exported ZIP for analysis. Default promotion and P5.3 closure require evidence review."
    )
except BaseException:
    (evidence / "integration-failure.txt").write_text(traceback.format_exc())
    print("Integration incomplete; export the failure and partial evidence.")

In [ ]:
try:
    (evidence / "source-after.json").write_text(json.dumps(verify_source(), indent=2))
except BaseException:
    (evidence / "source-verification-failure.txt").write_text(traceback.format_exc())
(evidence / "commands.json").write_text(json.dumps(commands, indent=2))
(evidence / "notebook-run.json").write_text(
    json.dumps(
        {
            "snapshot_sha256": SNAPSHOT_SHA256,
            "base_revision": identity["base_revision"],
            "scope": "Experimental FlashInfer candidate; not default promotion or full release",
            "environment_overrides": {
                key: environment[key]
                for key in (
                    "CUDA_VISIBLE_DEVICES",
                    "CUBLAS_WORKSPACE_CONFIG",
                    "OMP_NUM_THREADS",
                    "MKL_NUM_THREADS",
                )
            },
        },
        indent=2,
    )
)
files = {
    path.relative_to(evidence).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in evidence.rglob("*")
    if path.is_file()
}
(evidence / "checksums.json").write_text(json.dumps(files, indent=2))
archive_path = working / (run_root.name + ".zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("source-snapshot.zip", payload)
    for path in evidence.rglob("*"):
        if path.is_file():
            archive.write(path, "evidence/" + path.relative_to(evidence).as_posix())
print("DOWNLOAD", archive_path, "bytes", archive_path.stat().st_size, flush=True)
display(FileLink(os.path.relpath(archive_path, Path.cwd())))